# 03 Production: Advanced Modeling and Ensemble

実運用CSVを内部 split し、複数の前処理・複数モデル・波長範囲選択・VIP選択・Adversarial validation・Local model・Stacking / Ensemble を検討する本格モデル検討 notebook です。

`02_spectral_modeling_ch11_production.ipynb` は PLS 単一モデルの標準運用版、この `03` は改善余地を探索する実験版、という棲み分けです。重い処理は `RUN_*` フラグで明示的にオンにします。

## 0. 設定と共通関数

In [ ]:

from __future__ import annotations

from pathlib import Path
import json
import math
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
os.environ.setdefault('MPLCONFIGDIR', str(Path('/private/tmp') / 'matplotlib-cache'))
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter, find_peaks
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import KFold, GroupKFold, GroupShuffleSplit, GridSearchCV, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.utils.validation import check_array, check_is_fitted

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')

for font_name in ['Hiragino Sans', 'AppleGothic', 'Apple SD Gothic Neo', 'Yu Gothic', 'Meiryo', 'Noto Sans CJK JP']:
    try:
        plt.rcParams['font.family'] = font_name
        break
    except Exception:
        pass
plt.rcParams['axes.unicode_minus'] = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path('/Users/ogawatomohiro/myproject')
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'production'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 実運用ではここを実データCSVに変更してください。存在しない場合だけサンプルの train.csv で動作確認します。
DATA_PATH = DATA_DIR / 'production_dataset.csv'
SAMPLE_FALLBACK_PATH = DATA_DIR / 'train.csv'
ENCODINGS = ('cp932', 'utf-8-sig', 'utf-8', 'shift_jis')

TARGET_CANDIDATES = ['含水率', 'mc', 'moisture', 'moisture_content', 'water_content', 'target', 'y']
SAMPLE_CANDIDATES = ['サンプル名', 'サンプルID', 'sample name', 'sample_name', 'sample number', 'sample_id', 'id', 'ID']
DATE_CANDIDATES = ['日付', '測定日', 'date', 'measurement_date', 'measured_at']
GROUP_CANDIDATES = ['ロット', 'lot', 'batch', '樹種', 'species', 'species number', 'sample_name']

# 波長列は 900-1700 nm を主対象にします。実データが範囲外まで含む場合は None にしてください。
WAVELENGTH_RANGE_NM = (900, 1700)
VALID_SIZE = 0.20
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_SPLITS = 5
GROUP_SPLIT_COL = None  # 例: '日付' や 'ロット'。None の場合は目的変数の分布を保つ通常split。


def read_csv_with_fallback(path: Path, encodings=ENCODINGS) -> pd.DataFrame:
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def pick_first_existing(columns, candidates, required=False, label='column'):
    normalized = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in normalized:
            return normalized[key]
    if required:
        raise ValueError(f'{label} not found. candidates={candidates}')
    return None


def parse_wavelength_from_col(col):
    text = str(col).strip()
    if text in {'', 'nan'}:
        return None
    # Examples: 900, 900.0, 900nm, wl_900, Abs_900, absorbance_900_nm
    nums = re.findall(r'(?<!\d)(\d{3,5}(?:\.\d+)?)(?!\d)', text)
    if not nums:
        return None
    values = [float(x) for x in nums]
    # Prefer values in the expected nm range when multiple numbers exist.
    lo, hi = WAVELENGTH_RANGE_NM if WAVELENGTH_RANGE_NM is not None else (0, float('inf'))
    in_range = [v for v in values if lo <= v <= hi]
    return in_range[-1] if in_range else values[-1]


def detect_spectral_columns(df: pd.DataFrame, exclude_cols=None, wavelength_range_nm=WAVELENGTH_RANGE_NM):
    exclude = set(exclude_cols or [])
    records = []
    for col in df.columns:
        if col in exclude:
            continue
        wl = parse_wavelength_from_col(col)
        if wl is None:
            continue
        numeric = pd.to_numeric(df[col], errors='coerce')
        non_na_ratio = numeric.notna().mean()
        if non_na_ratio < 0.90:
            continue
        records.append((col, wl))
    if not records:
        raise ValueError('波長列を検出できませんでした。列名に 900, 901nm, Abs_900 などの波長値が含まれているか確認してください。')
    spec_info = pd.DataFrame(records, columns=['column', 'wavelength_nm']).drop_duplicates('column')
    if wavelength_range_nm is not None:
        lo, hi = wavelength_range_nm
        ranged = spec_info.query('@lo <= wavelength_nm <= @hi').copy()
        if len(ranged) >= 10:
            spec_info = ranged
    spec_info = spec_info.sort_values('wavelength_nm').reset_index(drop=True)
    return spec_info['column'].tolist(), spec_info


def make_y_bins(y, max_bins=8):
    y = pd.Series(y).astype(float)
    n_bins = min(max_bins, max(2, len(y) // 10))
    try:
        bins = pd.qcut(y, q=n_bins, duplicates='drop')
        counts = bins.value_counts()
        if len(counts) >= 2 and counts.min() >= 2:
            return bins.astype(str)
    except Exception:
        pass
    return None


def split_single_dataset(df, target_col, group_col=None, valid_size=VALID_SIZE, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    df = df.dropna(subset=[target_col]).reset_index(drop=True).copy()
    if group_col is not None and group_col in df.columns and df[group_col].nunique() >= 3:
        splitter1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_valid_idx, test_idx = next(splitter1.split(df, groups=df[group_col]))
        train_valid = df.iloc[train_valid_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)
        rel_valid = valid_size / (1 - test_size)
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=rel_valid, random_state=random_state)
        train_idx, valid_idx = next(splitter2.split(train_valid, groups=train_valid[group_col]))
        return train_valid.iloc[train_idx].reset_index(drop=True), train_valid.iloc[valid_idx].reset_index(drop=True), test_df

    stratify = make_y_bins(df[target_col])
    train_valid, test_df = train_test_split(df, test_size=test_size, random_state=random_state, shuffle=True, stratify=stratify)
    rel_valid = valid_size / (1 - test_size)
    stratify_tv = make_y_bins(train_valid[target_col])
    train_df, valid_df = train_test_split(train_valid, test_size=rel_valid, random_state=random_state, shuffle=True, stratify=stratify_tv)
    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred):
    return {
        'r2': float(r2_score(y_true, y_pred)),
        'rmse': rmse(y_true, y_pred),
        'mae': float(mean_absolute_error(y_true, y_pred)),
    }


def odd_window(window_length, n_points):
    w = int(window_length)
    if w % 2 == 0:
        w += 1
    max_w = n_points if n_points % 2 == 1 else n_points - 1
    w = max(3, min(w, max_w))
    return w


def load_production_dataset():
    path = DATA_PATH if DATA_PATH.exists() else SAMPLE_FALLBACK_PATH
    if path == SAMPLE_FALLBACK_PATH:
        print(f'NOTE: {DATA_PATH} がないため、動作確認用に {SAMPLE_FALLBACK_PATH} を読み込みます。')
    df = read_csv_with_fallback(path)
    target_col = pick_first_existing(df.columns, TARGET_CANDIDATES, required=True, label='target column')
    sample_col = pick_first_existing(df.columns, SAMPLE_CANDIDATES, required=False, label='sample id column')
    date_col = pick_first_existing(df.columns, DATE_CANDIDATES, required=False, label='date column')
    group_col = GROUP_SPLIT_COL or pick_first_existing(df.columns, GROUP_CANDIDATES, required=False, label='diagnostic group column')
    exclude_cols = [c for c in [target_col, sample_col, date_col, group_col] if c is not None]
    spectral_cols, spec_info = detect_spectral_columns(df, exclude_cols=exclude_cols)
    df[spectral_cols] = df[spectral_cols].apply(pd.to_numeric, errors='coerce')
    if date_col is not None:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    train_df, valid_df, test_df = split_single_dataset(df, target_col=target_col, group_col=GROUP_SPLIT_COL)
    config = {
        'data_path': str(path),
        'target_col': target_col,
        'sample_col': sample_col,
        'date_col': date_col,
        'diagnostic_group_col': group_col,
        'group_split_col': GROUP_SPLIT_COL,
        'n_spectral_cols': len(spectral_cols),
        'wavelength_min_nm': float(spec_info['wavelength_nm'].min()),
        'wavelength_max_nm': float(spec_info['wavelength_nm'].max()),
        'train_shape': train_df.shape,
        'valid_shape': valid_df.shape,
        'test_shape': test_df.shape,
    }
    return df, train_df, valid_df, test_df, spectral_cols, spec_info, config


def get_xy(part_df, spectral_cols, target_col):
    return part_df[spectral_cols].astype(float), part_df[target_col].astype(float)


def sample_ids(df, sample_col):
    if sample_col is not None and sample_col in df.columns:
        return df[sample_col].values
    return np.arange(len(df))


class SavitzkyGolayTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, window_length=21, polyorder=2, deriv=0, mode='interp'):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv
        self.mode = mode

    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.window_length_ = odd_window(self.window_length, self.n_features_in_)
        self.polyorder_ = min(int(self.polyorder), self.window_length_ - 1)
        return self

    def transform(self, X):
        check_is_fitted(self, ['n_features_in_', 'window_length_', 'polyorder_'])
        X = check_array(X, dtype=float)
        return savgol_filter(X, window_length=self.window_length_, polyorder=self.polyorder_, deriv=self.deriv, axis=1, mode=self.mode)


class SNVTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, ['n_features_in_'])
        X = check_array(X, dtype=float)
        mean = X.mean(axis=1, keepdims=True)
        std = X.std(axis=1, keepdims=True)
        std[std == 0] = 1.0
        return (X - mean) / std


class MSCTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.reference_ = X.mean(axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self, ['reference_'])
        X = check_array(X, dtype=float)
        corrected = np.empty_like(X, dtype=float)
        ref = self.reference_
        for i, row in enumerate(X):
            slope, intercept = np.polyfit(ref, row, deg=1)
            if abs(slope) < 1e-12:
                corrected[i] = row - intercept
            else:
                corrected[i] = (row - intercept) / slope
        return corrected


class AdaptivePLSRegression(BaseEstimator, RegressorMixin):
    def __init__(self, n_components=10, scale=True, max_iter=500, tol=1e-06):
        self.n_components = n_components
        self.scale = scale
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, X, y):
        X = check_array(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_components = max(1, min(int(self.n_components), X.shape[0] - 1, X.shape[1]))
        self.n_components_ = n_components
        self.model_ = PLSRegression(n_components=n_components, scale=self.scale, max_iter=self.max_iter, tol=self.tol)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        check_is_fitted(self, ['model_'])
        return self.model_.predict(X).ravel()


## 1. データ読み込み

In [ ]:
df, train_df, valid_df, test_df, spectral_cols, spec_info, config = load_production_dataset()
print(json.dumps(config, ensure_ascii=False, indent=2, default=str))
X_train, y_train = get_xy(train_df, spectral_cols, config['target_col'])
X_valid, y_valid = get_xy(valid_df, spectral_cols, config['target_col'])
X_test, y_test = get_xy(test_df, spectral_cols, config['target_col'])
sample_col = config['sample_col']
group_col = config['diagnostic_group_col']
wavelengths = spec_info['wavelength_nm'].to_numpy(float)

## 2. 実験フラグと候補定義

`RUN_STAGE_A` は軽量な標準比較、`RUN_STAGE_B` 以降は重い追加検討です。まずは Stage A と Stacking だけで動かし、必要に応じて個別フラグを `True` にします。

In [ ]:
RUN_STAGE_A = True
RUN_STAGE_B = False
RUN_INTERVAL_SELECTION = False
RUN_VIP_SELECTION = False
RUN_ADVERSARIAL_VALIDATION = False
RUN_LOCAL_MODEL = False
RUN_STACKING = True

preprocessing_candidates = {
    'raw': [],
    'savgol_smooth_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=0))],
    'savgol_1st_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=1))],
    'savgol_2nd_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
    'savgol_2nd_41': [('savgol', SavitzkyGolayTransformer(window_length=41, polyorder=2, deriv=2))],
    'snv_savgol_2nd_21': [('snv', SNVTransformer()), ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
    'msc_savgol_2nd_21': [('msc', MSCTransformer()), ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
}

def make_model(model_key):
    if model_key == 'pls10':
        return AdaptivePLSRegression(n_components=10, scale=True)
    if model_key == 'pls20':
        return AdaptivePLSRegression(n_components=20, scale=True)
    if model_key == 'ridge':
        return Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=10.0))])
    if model_key == 'elasticnet':
        return Pipeline([('scaler', StandardScaler()), ('elasticnet', ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=20000, random_state=RANDOM_STATE))])
    if model_key == 'svr':
        return Pipeline([('scaler', StandardScaler()), ('svr', SVR(C=10.0, epsilon=0.1, gamma='scale'))])
    if model_key == 'rf':
        return RandomForestRegressor(n_estimators=400, min_samples_leaf=3, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1)
    if model_key == 'extratrees':
        return ExtraTreesRegressor(n_estimators=400, min_samples_leaf=2, max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1)
    if model_key == 'gbr':
        return GradientBoostingRegressor(n_estimators=300, learning_rate=0.03, max_depth=3, min_samples_leaf=3, random_state=RANDOM_STATE)
    raise KeyError(model_key)

def build_pipeline(prep_name, model_key):
    steps = list(preprocessing_candidates[prep_name])
    model = make_model(model_key)
    if isinstance(model, Pipeline):
        steps.extend([(f'{model_key}_{name}', step) for name, step in model.steps])
    else:
        steps.append((model_key, model))
    return Pipeline(steps)

stage_a_model_keys = ['pls10', 'pls20', 'ridge', 'elasticnet', 'svr']
stage_b_model_keys = ['rf', 'extratrees', 'gbr']

candidate_keys = []
if RUN_STAGE_A:
    candidate_keys += [(prep, model) for prep in preprocessing_candidates for model in stage_a_model_keys]
if RUN_STAGE_B:
    candidate_keys += [(prep, model) for prep in ['raw', 'savgol_2nd_21', 'snv_savgol_2nd_21'] for model in stage_b_model_keys]

if not candidate_keys:
    raise ValueError('At least one of RUN_STAGE_A or RUN_STAGE_B must be True for the main model comparison.')

candidate_registry = {f'{prep}__{model}': build_pipeline(prep, model) for prep, model in candidate_keys}

def get_candidate_estimator(model_name):
    if model_name not in candidate_registry:
        raise KeyError(f'Candidate estimator is not registered: {model_name}')
    return clone(candidate_registry[model_name])

flag_summary = pd.DataFrame([
    {'flag': 'RUN_STAGE_A', 'enabled': RUN_STAGE_A, 'purpose': '主要前処理 x PLS/Ridge/ElasticNet/SVR'},
    {'flag': 'RUN_STAGE_B', 'enabled': RUN_STAGE_B, 'purpose': '木系・Boosting系モデル'},
    {'flag': 'RUN_INTERVAL_SELECTION', 'enabled': RUN_INTERVAL_SELECTION, 'purpose': '波長区間ごとのモデル評価'},
    {'flag': 'RUN_VIP_SELECTION', 'enabled': RUN_VIP_SELECTION, 'purpose': 'PLS VIPによる波長選択'},
    {'flag': 'RUN_ADVERSARIAL_VALIDATION', 'enabled': RUN_ADVERSARIAL_VALIDATION, 'purpose': 'train/test分布差の診断'},
    {'flag': 'RUN_LOCAL_MODEL', 'enabled': RUN_LOCAL_MODEL, 'purpose': '近傍サンプルだけで予測するLocal PLS'},
    {'flag': 'RUN_STACKING', 'enabled': RUN_STACKING, 'purpose': 'OOF予測を使ったStacking/Ensemble'},
])
display(flag_summary)
print('main candidates:', len(candidate_keys))


## 3. Stage A/B: KFold OOF評価

`RUN_STAGE_A` と `RUN_STAGE_B` で有効化した候補を、train 内のKFold OOFで評価します。OOF予測は後段のアンサンブルやStackingの材料になります。

In [ ]:
def evaluate_cv(estimator, X, y, model_name, cv=None):
    cv = cv or KFold(n_splits=min(N_SPLITS, len(X)), shuffle=True, random_state=RANDOM_STATE)
    oof = np.full(len(y), np.nan, dtype=float)
    fold_rows = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        est = clone(estimator)
        est.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = np.asarray(est.predict(X.iloc[va_idx])).ravel()
        oof[va_idx] = pred
        fold_rows.append({'model_name': model_name, 'fold': fold, 'n_train': len(tr_idx), 'n_valid': len(va_idx), **regression_metrics(y.iloc[va_idx], pred)})
    score = {'model_name': model_name, **regression_metrics(y, oof)}
    oof_df_part = pd.DataFrame({'sample_id': sample_ids(train_df, sample_col), 'model_name': model_name, 'y_true': y.values, 'oof_pred': oof})
    if group_col is not None and group_col in train_df.columns:
        oof_df_part['diagnostic_group'] = train_df[group_col].astype(str).values
    return score, pd.DataFrame(fold_rows), oof_df_part

score_rows = []
fold_parts = []
oof_parts = []
main_candidate_items = list(candidate_registry.items())
for i, (name, estimator_template) in enumerate(main_candidate_items, start=1):
    print(f'[{i:02d}/{len(main_candidate_items)}] {name}')
    try:
        score, fold_df, oof_part = evaluate_cv(clone(estimator_template), X_train, y_train, name)
        score_rows.append(score)
        fold_parts.append(fold_df)
        oof_parts.append(oof_part)
    except Exception as exc:
        score_rows.append({'model_name': name, 'r2': np.nan, 'rmse': np.nan, 'mae': np.nan, 'error': repr(exc)})

scores_df = pd.DataFrame(score_rows).sort_values('rmse')
fold_scores_df = pd.concat(fold_parts, ignore_index=True) if fold_parts else pd.DataFrame()
oof_df = pd.concat(oof_parts, ignore_index=True) if oof_parts else pd.DataFrame()
scores_df.to_csv(OUTPUT_DIR / '03_model_scores_cv.csv', index=False, encoding='utf-8-sig')
fold_scores_df.to_csv(OUTPUT_DIR / '03_fold_scores_cv.csv', index=False, encoding='utf-8-sig')
oof_df.to_csv(OUTPUT_DIR / '03_oof_predictions.csv', index=False, encoding='utf-8-sig')
display(scores_df.head(20))

## 4. 任意: 波長区間選択

`RUN_INTERVAL_SELECTION = True` のとき、波長範囲を複数区間に分割し、区間ごとの PLS/Ridge/SVR を評価します。どの波長帯が効いているかの診断と、区間モデルの候補作りに使います。

In [ ]:
class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        self.columns_ = np.asarray(self.columns, dtype=int)
        return self

    def transform(self, X):
        X = check_array(X, dtype=float)
        return X[:, self.columns_]


def make_wavelength_intervals(wavelengths, n_intervals):
    order = np.argsort(np.asarray(wavelengths, dtype=float))
    return [chunk for chunk in np.array_split(order, n_intervals) if len(chunk) >= 5]

if RUN_INTERVAL_SELECTION:
    interval_score_rows = []
    interval_fold_parts = []
    interval_oof_parts = []
    interval_candidates = []
    for n_intervals in [4, 6, 8]:
        for interval_id, cols in enumerate(make_wavelength_intervals(wavelengths, n_intervals), start=1):
            wn_min = float(np.min(wavelengths[cols]))
            wn_max = float(np.max(wavelengths[cols]))
            for model_key in ['pls10', 'ridge', 'svr']:
                name = f'interval{n_intervals}_{interval_id}_{wn_min:.0f}_{wn_max:.0f}__{model_key}'
                est = Pipeline([
                    ('select_interval', ColumnSelector(cols)),
                    ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2)),
                    ('model', make_model(model_key)),
                ])
                interval_candidates.append((name, est))
                candidate_registry[name] = est
    for i, (name, est) in enumerate(interval_candidates, start=1):
        print(f'[interval {i:02d}/{len(interval_candidates)}] {name}')
        score, fold_df, oof_part = evaluate_cv(est, X_train, y_train, name)
        interval_score_rows.append(score)
        interval_fold_parts.append(fold_df)
        interval_oof_parts.append(oof_part)
    interval_scores_df = pd.DataFrame(interval_score_rows).sort_values('rmse')
    interval_scores_df.to_csv(OUTPUT_DIR / '03_interval_selection_scores_cv.csv', index=False, encoding='utf-8-sig')
    display(interval_scores_df.head(20))
    scores_df = pd.concat([scores_df, interval_scores_df], ignore_index=True).sort_values('rmse')
    fold_scores_df = pd.concat([fold_scores_df, *interval_fold_parts], ignore_index=True)
    oof_df = pd.concat([oof_df, *interval_oof_parts], ignore_index=True)
else:
    interval_scores_df = pd.DataFrame()


## 5. 任意: VIPによる波長選択

`RUN_VIP_SELECTION = True` のとき、PLSのVIP scoreで波長を選択する候補をPipeline内で評価します。特徴量選択は各foldのtrainだけでfitされます。

In [ ]:
def calculate_vip(pls_model):
    t = pls_model.x_scores_
    w = pls_model.x_weights_
    q = pls_model.y_loadings_.reshape(-1, 1)
    p, h = w.shape
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    if total_s <= 0:
        return np.ones(p)
    vip = np.sqrt(p * (w ** 2 @ s).ravel() / total_s)
    return vip


class VIPSelector(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=10, threshold=1.0, max_features=None):
        self.n_components = n_components
        self.threshold = threshold
        self.max_features = max_features

    def fit(self, X, y):
        X = check_array(X, dtype=float)
        n_components = max(1, min(int(self.n_components), X.shape[0] - 1, X.shape[1]))
        self.pls_ = PLSRegression(n_components=n_components, scale=True)
        self.pls_.fit(X, y)
        vip = calculate_vip(self.pls_)
        support = vip >= self.threshold
        if self.max_features is not None:
            top_idx = np.argsort(vip)[-min(int(self.max_features), X.shape[1]):]
            top_support = np.zeros(X.shape[1], dtype=bool)
            top_support[top_idx] = True
            support = support & top_support
        if support.sum() < 3:
            support[np.argsort(vip)[-min(10, X.shape[1]):]] = True
        self.vip_scores_ = vip
        self.support_ = support
        return self

    def transform(self, X):
        check_is_fitted(self, ['support_'])
        X = check_array(X, dtype=float)
        return X[:, self.support_]

if RUN_VIP_SELECTION:
    vip_candidates = []
    for threshold in [0.8, 1.0, 1.2]:
        for model_key in ['pls10', 'ridge', 'svr']:
            name = f'vip{threshold:g}__{model_key}'
            est = Pipeline([
                ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2)),
                ('vip', VIPSelector(n_components=10, threshold=threshold, max_features=300)),
                ('model', make_model(model_key)),
            ])
            vip_candidates.append((name, est))
            candidate_registry[name] = est
    vip_score_rows = []
    vip_fold_parts = []
    vip_oof_parts = []
    for i, (name, est) in enumerate(vip_candidates, start=1):
        print(f'[vip {i:02d}/{len(vip_candidates)}] {name}')
        score, fold_df, oof_part = evaluate_cv(est, X_train, y_train, name)
        vip_score_rows.append(score)
        vip_fold_parts.append(fold_df)
        vip_oof_parts.append(oof_part)
    vip_scores_df = pd.DataFrame(vip_score_rows).sort_values('rmse')
    vip_scores_df.to_csv(OUTPUT_DIR / '03_vip_selection_scores_cv.csv', index=False, encoding='utf-8-sig')
    display(vip_scores_df)
    scores_df = pd.concat([scores_df, vip_scores_df], ignore_index=True).sort_values('rmse')
    fold_scores_df = pd.concat([fold_scores_df, *vip_fold_parts], ignore_index=True)
    oof_df = pd.concat([oof_df, *vip_oof_parts], ignore_index=True)
else:
    vip_scores_df = pd.DataFrame()


## 6. 任意: Adversarial validation

`RUN_ADVERSARIAL_VALIDATION = True` のとき、train と test holdout を分類できるかを確認します。AUCが高い場合は、split間の分布差や測定日の偏りを疑います。

In [ ]:
if RUN_ADVERSARIAL_VALIDATION:
    X_adv = pd.concat([X_train, X_test], ignore_index=True)
    y_adv = np.r_[np.zeros(len(X_train), dtype=int), np.ones(len(X_test), dtype=int)]
    cv_adv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    adv_models = {
        'logistic': Pipeline([
            ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2)),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=5000, class_weight='balanced')),
        ]),
        'random_forest': Pipeline([
            ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2)),
            ('clf', RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')),
        ]),
    }
    adv_rows = []
    for name, clf in adv_models.items():
        proba = cross_val_predict(clf, X_adv, y_adv, cv=cv_adv, method='predict_proba')[:, 1]
        adv_rows.append({'model_name': name, 'auc_train_vs_test': float(roc_auc_score(y_adv, proba))})
    adversarial_scores_df = pd.DataFrame(adv_rows).sort_values('auc_train_vs_test', ascending=False)
    adversarial_scores_df.to_csv(OUTPUT_DIR / '03_adversarial_validation_scores.csv', index=False, encoding='utf-8-sig')
    display(adversarial_scores_df)
else:
    adversarial_scores_df = pd.DataFrame()


## 7. 任意: Local PLS

`RUN_LOCAL_MODEL = True` のとき、PCA空間で近いtrainサンプルだけを使い、validサンプルごとにPLSをfitします。計算は重めですが、分布が局所的に異なるデータで有効な場合があります。

In [ ]:
def predict_local_pls(X_reference, y_reference, X_query, k=100, n_components=10):
    preprocessing = Pipeline([
        ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2)),
        ('scaler', StandardScaler()),
    ])
    X_ref_proc = preprocessing.fit_transform(X_reference, y_reference)
    X_query_proc = preprocessing.transform(X_query)
    pca_components = max(2, min(20, X_ref_proc.shape[0] - 1, X_ref_proc.shape[1]))
    pca = PCA(n_components=pca_components, random_state=RANDOM_STATE)
    ref_scores = pca.fit_transform(X_ref_proc)
    query_scores = pca.transform(X_query_proc)
    preds = []
    for row, query_raw in zip(query_scores, X_query_proc):
        distances = np.sqrt(((ref_scores - row) ** 2).sum(axis=1))
        idx = np.argsort(distances)[:min(k, len(y_reference))]
        model = AdaptivePLSRegression(n_components=n_components, scale=True)
        model.fit(X_ref_proc[idx], np.asarray(y_reference)[idx])
        preds.append(float(model.predict(query_raw.reshape(1, -1))[0]))
    return np.asarray(preds)

if RUN_LOCAL_MODEL:
    local_rows = []
    local_valid_pred_map = {}
    for k in [50, 100, 200]:
        print(f'Local PLS k={k}')
        pred = predict_local_pls(X_train.values, y_train.values, X_valid.values, k=k, n_components=10)
        name = f'local_pls_k{k}'
        local_valid_pred_map[name] = pred
        local_rows.append({'model_name': name, **regression_metrics(y_valid, pred)})
    local_scores_df = pd.DataFrame(local_rows).sort_values('rmse')
    local_scores_df.to_csv(OUTPUT_DIR / '03_local_model_valid_scores.csv', index=False, encoding='utf-8-sig')
    display(local_scores_df)
else:
    local_scores_df = pd.DataFrame()
    local_valid_pred_map = {}


## 8. valid確認と誤差診断

CV上位候補をvalidで確認します。Interval/VIP候補を有効化した場合は、それらも `scores_df` に統合されたうえで候補に入ります。

In [ ]:
valid_rows = []
valid_pred_map = {}
for name in scores_df.dropna(subset=['rmse']).head(12)['model_name']:
    est = get_candidate_estimator(name)
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_valid)).ravel()
    valid_pred_map[name] = pred
    valid_rows.append({'model_name': name, 'cv_rmse': scores_df.loc[scores_df['model_name'] == name, 'rmse'].iloc[0], **regression_metrics(y_valid, pred)})
valid_scores = pd.DataFrame(valid_rows).sort_values('rmse')
valid_scores.to_csv(OUTPUT_DIR / '03_valid_scores.csv', index=False, encoding='utf-8-sig')
display(valid_scores)

best_valid_name = valid_scores.iloc[0]['model_name']
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(y_valid, valid_pred_map[best_valid_name], s=26, alpha=0.75)
lo = min(y_valid.min(), valid_pred_map[best_valid_name].min())
hi = max(y_valid.max(), valid_pred_map[best_valid_name].max())
pad = (hi - lo) * 0.05 if hi > lo else 1
axes[0].plot([lo - pad, hi + pad], [lo - pad, hi + pad], color='black', linestyle='--')
axes[0].set_xlabel('Measured')
axes[0].set_ylabel('Predicted')
axes[0].set_title(best_valid_name)
resid = y_valid.values - valid_pred_map[best_valid_name]
axes[1].scatter(valid_pred_map[best_valid_name], resid, s=26, alpha=0.75)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '03_valid_best_diagnostics.png', dpi=160)

if group_col is not None and group_col in train_df.columns:
    tmp = oof_df[oof_df['model_name'] == scores_df.iloc[0]['model_name']].copy()
    tmp['abs_error'] = (tmp['y_true'] - tmp['oof_pred']).abs()
    display(tmp.groupby('diagnostic_group').agg(n=('abs_error', 'size'), mae=('abs_error', 'mean'), y_mean=('y_true', 'mean')).sort_values('mae', ascending=False).head(20))

## 9. アンサンブル / Stacking

`RUN_STACKING = True` のとき、OOF予測を横持ち化して上位モデルの平均・重み付き平均を評価します。

In [ ]:
def ensemble_oof_scores(oof_df, scores_df, top_ns=(3, 5, 8)):
    base = oof_df.dropna(subset=['oof_pred']).copy()
    pivot = base.pivot_table(index='sample_id', columns='model_name', values='oof_pred', aggfunc='first')
    y_map = base.drop_duplicates('sample_id').set_index('sample_id')['y_true']
    rows = []
    pred_map = {}
    for n in top_ns:
        top = [m for m in scores_df.dropna(subset=['rmse']).head(n)['model_name'] if m in pivot.columns]
        if len(top) < 2:
            continue
        pred_mean = pivot[top].mean(axis=1)
        rows.append({'model_name': f'ensemble_mean_top{len(top)}', 'members': ','.join(top), **regression_metrics(y_map.loc[pred_mean.index], pred_mean)})
        pred_map[f'ensemble_mean_top{len(top)}'] = (top, None)
        weights = 1 / scores_df.set_index('model_name').loc[top, 'rmse'].clip(lower=1e-9)
        weights = weights / weights.sum()
        pred_weighted = pivot[top].mul(weights, axis=1).sum(axis=1)
        rows.append({'model_name': f'ensemble_weighted_top{len(top)}', 'members': ','.join(top), **regression_metrics(y_map.loc[pred_weighted.index], pred_weighted)})
        pred_map[f'ensemble_weighted_top{len(top)}'] = (top, weights)
    result = pd.DataFrame(rows)
    if result.empty:
        result = pd.DataFrame(columns=['model_name', 'members', 'r2', 'rmse', 'mae'])
        return result, pred_map
    return result.sort_values('rmse'), pred_map

if RUN_STACKING:
    ensemble_scores, ensemble_recipe = ensemble_oof_scores(oof_df, scores_df)
    ensemble_scores.to_csv(OUTPUT_DIR / '03_ensemble_scores_cv.csv', index=False, encoding='utf-8-sig')
    display(ensemble_scores)
else:
    ensemble_scores = pd.DataFrame()
    ensemble_recipe = {}

all_valid_scores = pd.concat([valid_scores.assign(kind='single')], ignore_index=True)
valid_ensemble_rows = []
for ens_name, (members, weights) in ensemble_recipe.items():
    member_preds = pd.DataFrame({m: valid_pred_map[m] for m in members if m in valid_pred_map})
    if member_preds.shape[1] != len(members):
        continue
    if weights is None:
        pred = member_preds.mean(axis=1).values
    else:
        pred = member_preds[members].mul(weights.loc[members], axis=1).sum(axis=1).values
    valid_ensemble_rows.append({'model_name': ens_name, 'kind': 'ensemble', **regression_metrics(y_valid, pred)})
valid_ensemble_scores = pd.DataFrame(valid_ensemble_rows)
if not valid_ensemble_scores.empty:
    valid_ensemble_scores = valid_ensemble_scores.sort_values('rmse')
display(valid_ensemble_scores)

## 10. test 最終評価

validで選んだ単一モデルまたはアンサンブルを、`train + valid` で再学習してからtestで独立評価します。

In [ ]:
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
X_train_valid, y_train_valid = get_xy(train_valid_df, spectral_cols, config['target_col'])

# validで良いものを優先。アンサンブルがvalidで改善していれば採用します。
if not valid_ensemble_scores.empty and valid_ensemble_scores.iloc[0]['rmse'] < valid_scores.iloc[0]['rmse']:
    selected_name = valid_ensemble_scores.iloc[0]['model_name']
    members, weights = ensemble_recipe[selected_name]
    test_member_preds = {}
    fitted_members = {}
    for m in members:
        est = get_candidate_estimator(m)
        est.fit(X_train_valid, y_train_valid)
        fitted_members[m] = est
        test_member_preds[m] = np.asarray(est.predict(X_test)).ravel()
    test_member_df = pd.DataFrame(test_member_preds)
    if weights is None:
        test_pred = test_member_df.mean(axis=1).values
    else:
        test_pred = test_member_df[members].mul(weights.loc[members], axis=1).sum(axis=1).values
    final_artifact = {'kind': 'ensemble', 'name': selected_name, 'members': fitted_members, 'weights': None if weights is None else weights.to_dict()}
else:
    selected_name = valid_scores.iloc[0]['model_name']
    final_estimator = get_candidate_estimator(selected_name)
    final_estimator.fit(X_train_valid, y_train_valid)
    test_pred = np.asarray(final_estimator.predict(X_test)).ravel()
    final_artifact = {'kind': 'single', 'name': selected_name, 'model': final_estimator}

test_metrics = regression_metrics(y_test, test_pred)
print(json.dumps({'selected_model': selected_name, **test_metrics}, ensure_ascii=False, indent=2))

prediction_df = pd.DataFrame({
    'sample_id': sample_ids(test_df, sample_col),
    'split': 'test',
    'measured_moisture': y_test.values,
    'predicted_moisture': test_pred,
    'residual': y_test.values - test_pred,
})
if group_col is not None and group_col in test_df.columns:
    prediction_df['diagnostic_group'] = test_df[group_col].astype(str).values
prediction_df.to_csv(OUTPUT_DIR / '03_test_predictions.csv', index=False, encoding='utf-8-sig')
with open(OUTPUT_DIR / '03_final_model.pkl', 'wb') as f:
    pickle.dump({'artifact': final_artifact, 'spectral_cols': spectral_cols, 'config': config, 'spec_info': spec_info}, f)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, test_pred, s=28, alpha=0.75)
lo = min(y_test.min(), test_pred.min())
hi = max(y_test.max(), test_pred.max())
pad = (hi - lo) * 0.05 if hi > lo else 1
ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color='black', linestyle='--')
ax.set_xlim(lo - pad, hi + pad)
ax.set_ylim(lo - pad, hi + pad)
ax.set_xlabel('Measured moisture')
ax.set_ylabel('Predicted moisture')
ax.set_title(selected_name)
ax.text(0.04, 0.96, f"R2={test_metrics['r2']:.3f}\nRMSE={test_metrics['rmse']:.3f}\nMAE={test_metrics['mae']:.3f}", transform=ax.transAxes, va='top', bbox={'facecolor': 'white', 'edgecolor': '#cccccc'})
fig.tight_layout()
fig.savefig(FIGURE_DIR / '03_test_measured_vs_predicted.png', dpi=160)
display(prediction_df.head())